# CommGuard benign corpus

This notebook restores reviewed calibration evidence and collects an 18-run training, inference, and hard-negative corpus. Collection is not a detector result and does not establish training-versus-inference performance.


## Install a reviewed source revision and restore calibration

Upload the calibration archive from the first notebook as a Kaggle Dataset and set `INPUT_BUNDLE` to its read-only `/kaggle/input/...` path. Restoration refuses to overwrite a non-empty artifact directory and rejects archive paths that escape the destination.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import importlib
import json
import site
import subprocess
import sys
import tarfile

REPO_URL = "https://github.com/waqasm86/CommGuard.git"
GIT_REF = "main"  # Replace with the same reviewed SHA used for calibration.
INPUT_BUNDLE = None  # Example: Path("/kaggle/input/commguard-calibration/bundle.tar.gz")
REPO = Path("/kaggle/working/CommGuard")
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")

if REPO.exists() and not (REPO / ".git").is_dir():
    raise RuntimeError(f"Refusing to replace non-Git directory: {REPO}")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps", "-e", str(REPO)],
    check=True,
)
site.addsitedir(str(REPO / "src"))
importlib.invalidate_caches()

def restore_bundle(bundle: Path, destination: Path) -> None:
    if destination.exists() and any(destination.iterdir()):
        raise RuntimeError(f"Refusing to overwrite non-empty artifact directory: {destination}")
    destination.mkdir(parents=True, exist_ok=True)
    destination_root = destination.resolve()
    with tarfile.open(bundle, "r:gz") as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            if target != destination_root and destination_root not in target.parents:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
            if not (member.isfile() or member.isdir()):
                raise RuntimeError(f"Unsupported archive member: {member.name}")
        archive.extractall(destination)

if INPUT_BUNDLE is not None:
    restore_bundle(Path(INPUT_BUNDLE), ARTIFACTS)
elif not ARTIFACTS.exists():
    raise RuntimeError("Set INPUT_BUNDLE to the calibration archive from notebook 1.")

calibration_paths = sorted((ARTIFACTS / "results").glob("calibration-*.json"))
if not calibration_paths:
    raise RuntimeError("The restored bundle contains no calibration result.")
calibration = json.loads(calibration_paths[-1].read_text(encoding="utf-8"))
if calibration["status"] == "not_supported":
    raise RuntimeError("Calibration is not_supported; corpus escalation is blocked.")
print("Reviewed source commit:", COMMIT)
print("Calibration status:", calibration["status"])


## Strict dual-T4 gate

The source revision should match the calibration archive provenance. Strict preflight again verifies exactly two NVIDIA T4 GPUs before collection.


In [ ]:
from commguard.environment import check_environment

environment = check_environment(strict=True, output=ARTIFACTS)
assert environment["readiness"]["exactly_two_gpus"]
assert environment["readiness"]["both_t4"]
print("Strict dual-T4 readiness:", environment["strict_ready"])


## Execute the 18-run corpus

This GPU workload is disabled by default. Review the six workload families, set `RUN_BENIGN_CORPUS = True`, and rerun this cell. Failed runs remain recorded and are not hidden.


In [ ]:
from commguard.artifacts import ArtifactStore
from commguard.orchestrator import run_experiment

RUN_BENIGN_CORPUS = False
WORKLOADS = (
    "ddp_train",
    "inference_prefill_independent",
    "inference_synchronized",
    "control_compute",
    "control_host_transfer",
    "control_idle",
)
REPETITIONS = 3

outcomes = []
if RUN_BENIGN_CORPUS:
    for workload in WORKLOADS:
        for repetition in range(REPETITIONS):
            outcomes.append(
                run_experiment(
                    workload,
                    output=ARTIFACTS,
                    overrides={
                        "repetition": repetition,
                        "sampling_interval_s": 0.2,
                    },
                    raise_on_failure=False,
                )
            )
    summary = {
        "runs": len(outcomes),
        "completed": sum(
            outcome["manifest"]["exit_status"] == "completed" for outcome in outcomes
        ),
        "run_ids": [outcome["run_id"] for outcome in outcomes],
    }
    ArtifactStore(ARTIFACTS).write_json(
        "benign-corpus-summary.json", summary, validate=False
    )
    print(json.dumps(summary, indent=2))
else:
    print("Corpus skipped. Set RUN_BENIGN_CORPUS=True only after reviewing this cell.")


## Export the combined evidence

Download this archive from Kaggle's Output panel. Upload it as a Kaggle Dataset for `commguard_detector_evaluation.ipynb`, then set that notebook's `INPUT_BUNDLE` to the uploaded `/kaggle/input/...` archive.


In [ ]:
if RUN_BENIGN_CORPUS:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archive_path = Path("/kaggle/working") / f"commguard-corpus-{stamp}-{COMMIT[:12]}.tar.gz"
    exported = ArtifactStore(ARTIFACTS).export(archive_path)
    print("Download for the next notebook:", exported)
else:
    print("Nothing exported because corpus collection was not enabled.")
